# Citation Fidelity in Evidence Lab Briefs

**Central question: is each fact in a generated brief actually supported by text in a source document?**

This notebook reads an actual brief generated by Evidence Lab straight from the database,
extracts every *(citing sentence, cited source)* pair, and inspects the citation evidence the
system already stores — including the exact supporting text in the source document each
citation links to.

## What the system stores about citations

A brief lives in the Postgres table **`briefs`**, column **`content`** (JSONB). Each researched
section carries everything needed for fidelity analysis — no separate lookups required:

- `sections[].content` — the section's markdown. Facts cite sources inline with `[n]` /
  `[n, m]` markers (one marker per sentence; `n` is a **per-section** index).
- `sections[].sources[]` — one record per retrieved source chunk
  (`SourceReference` in `ui/frontend/src/types/api.ts`):
  - `index` — the number the `[n]` markers refer to,
  - `chunkId` / `docId` / `page` / `headings` / `bbox` — where the excerpt sits in the source PDF,
  - `title`, `pdfUrl`, `reportUrl` — the source document,
  - **`text` — the full, untruncated chunk text from the source document** (a leading
    `-- Heading > Subheading --` line is the chunk's heading breadcrumb),
  - **`claimMatches[]` — per citing sentence (`claim`), the LLM-selected character spans
    (`matches[].start/end/matchedText`) inside the excerpt that support that exact claim.**
    Offsets are relative to the excerpt *body* (after the breadcrumb line). These are computed
    by the UI via `POST /highlight` after research finishes and saved with the brief — so they
    are only present for sections whose highlight pass has run (they backfill lazily).

Two other places mirror citation data (not used here, noted for completeness):
- `user_activity` rows with `filters = {"type": "brief"}` store the brief markdown plus up to
  50 sources (`chunk_id`, `page_num`, `chunk_text`, `link`).
- The ground-truth chunk text lives in `chunks_<data_source>.sys_text` (Postgres) and the
  Qdrant collection `chunks_<data_source>` — used below to verify the stored excerpts.

## What this notebook does

1. Load a brief and show its structure.
2. Extract every citation pair and resolve it to its stored source excerpt.
3. Report fidelity signals already in the data: sentence-level citation coverage, dangling
   citation markers, and the stored per-claim supporting spans.
4. Verify the stored excerpts against the ground-truth chunk text in `chunks_<source>`.
5. Optionally re-run the system's own claim-support check (`POST /highlight`, the same
   endpoint the UI uses) for cited sentences that have no stored spans yet.
6. Quantify groundedness with **RAGAS Faithfulness**: an LLM judge decomposes each section
   into atomic claims and delivers a supported/unsupported verdict (with reason) for every
   claim against the cited source texts — turning "a human checks everything" into
   "a human reviews the flagged claims".

## Requirements

- The local Docker stack running (`docker compose up -d`) — Postgres on `localhost:5432`,
  API on `localhost:8000`.
- A dedicated Python 3.11 venv with `pip install -r notebooks/requirements.txt`
  (see that file's header — ragas conflicts with the app's langchain pins).
- For RAGAS (section 8): `AZURE_FOUNDRY_KEY` / `AZURE_FOUNDRY_ENDPOINT` in the repo `.env`
  (already used by the app; no new credentials).

Run from the repo (helpers import from `notebooks/citation_fidelity_lib.py` and
`pipeline/utilities/text_cleaning.py`).

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import psycopg2

pd.set_option("display.max_colwidth", 90)

# Repo root: parent of notebooks/ when run in place, else the cwd.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

from citation_fidelity_lib import (  # noqa: E402
    CitationPair,
    build_faithfulness_input,
    extract_cited_numbers,
    extract_citation_pairs,
    find_claim_match,
    normalize_claim_text,
    normalize_ws,
    parse_section_breadcrumb,
    sentences_citing,
    split_sentences,
)
from pipeline.utilities.text_cleaning import clean_text  # noqa: E402


def read_env(path: Path) -> dict:
    """Minimal KEY=VALUE parser for the repo .env (no dependency needed)."""
    env = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        env[key.strip()] = value.strip().strip('"').strip("'")
    return env


DOTENV = read_env(REPO_ROOT / ".env")

# .env holds container hostnames (POSTGRES_HOST=postgres); from the host we
# connect to the published ports on localhost. Override via environment.
PG = dict(
    host=os.environ.get("EVIDENCELAB_PG_HOST", "localhost"),
    port=int(os.environ.get("EVIDENCELAB_PG_PORT", "5432")),
    dbname=DOTENV.get("POSTGRES_DBNAME") or DOTENV.get("POSTGRES_DB", "evidencelab"),
    user=DOTENV["POSTGRES_USER"],
    password=DOTENV["POSTGRES_PASSWORD"],
)
API_BASE = os.environ.get("EVIDENCELAB_API_BASE", "http://localhost:8000")
API_KEY = DOTENV["REACT_APP_API_KEY"]

conn = psycopg2.connect(**PG)
print(f"Connected to postgres://{PG['user']}@{PG['host']}:{PG['port']}/{PG['dbname']}")

Connected to postgres://evidencelab@localhost:5432/evidencelab


## 1. Pick a brief

All briefs stored by the system, most recently updated first. By default the notebook analyses
the most recently updated brief that has at least one researched section — set `BRIEF_ID` to
analyse a specific one.

In [2]:
def researched_section_count(content: dict) -> int:
    return sum(
        1
        for s in content.get("sections", [])
        if s.get("status") == "done" and (s.get("content") or "").strip()
    )


with conn.cursor() as cur:
    cur.execute(
        """SELECT id, title, data_source, updated_at, content
           FROM briefs ORDER BY updated_at DESC"""
    )
    brief_rows = cur.fetchall()

overview = pd.DataFrame(
    [
        {
            "id": r[0],
            "title": r[1],
            "data_source": r[2],
            "updated_at": r[3],
            "sections": len(r[4].get("sections", [])),
            "researched_sections": researched_section_count(r[4]),
        }
        for r in brief_rows
    ]
)
display(overview)

# Set to a specific id to override the default selection.
BRIEF_ID = None

if BRIEF_ID is None:
    BRIEF_ID = next(r[0] for r in brief_rows if researched_section_count(r[4]) > 0)
brief_row = next(r for r in brief_rows if r[0] == BRIEF_ID)
brief_id, brief_title, data_source, _, brief = brief_row
if not data_source:
    raise ValueError(f"Brief {brief_id} has no data_source; set BRIEF_ID to another brief")
chunks_table = f"chunks_{data_source}"
print(f"Analysing brief {brief_id}\n  title: {brief_title}\n  data_source: {data_source} (ground truth: {chunks_table})")

,id,title,data_source,updated_at,sections,researched_sections
0,190ce56e-cda4-4b71-8d76-8e8e856df39b,"School Feeding Programmes: Evidence On Learning, Nutrition, And Local Procurement Outc...",wfp,2026-09-01 21:06:47.518387+00:00,6,4
1,8d0510b0-69ec-46d3-8e77-09a97e0e1923,Girls Education Kenya,wfp,2026-08-31 14:43:07.848860+00:00,3,0
2,c4bbbd5b-bde0-4561-89ef-15d21e11a300,Effects Of Covid 19 On WFP Community Engagement In Subsaharan Africa,wfp,2026-08-31 14:41:43.150347+00:00,6,4
3,35d9855c-7ebc-4955-a23d-602066246dfb,Community Engagement During Covid19,NaN,2026-08-10 20:53:48.491495+00:00,3,1
4,d0b979b8-65d6-41ed-a451-b34f4658596a,Community Engagement During Covid19,wfp,2026-08-10 19:16:40.058224+00:00,3,2
5,d9db62ce-a67f-48db-a54e-ca997de7e74e,Community Engagement During Covid19,NaN,2026-08-10 19:16:39.726085+00:00,3,2


Analysing brief 190ce56e-cda4-4b71-8d76-8e8e856df39b
  title: School Feeding Programmes: Evidence On Learning, Nutrition, And Local Procurement Outcomes
  data_source: wfp (ground truth: chunks_wfp)


## 2. Brief structure

Per section: the markdown length, number of `[n]` markers, retrieved sources, and how many of
those sources already carry stored `claimMatches` (the per-claim supporting spans).

In [3]:
sections = [
    s
    for s in brief["sections"]
    if s.get("status") == "done" and (s.get("content") or "").strip()
]

section_overview = pd.DataFrame(
    [
        {
            "section": s["title"],
            "chars": len(s.get("content") or ""),
            "sentences": len(split_sentences(s.get("content") or "")),
            "citation_markers": len(extract_cited_numbers(s.get("content") or "")),
            "sources": len(s.get("sources") or []),
            "sources_with_claimMatches": sum(
                1 for src in s.get("sources") or [] if src.get("claimMatches")
            ),
        }
        for s in sections
    ]
)
display(section_overview)
skipped = [s["title"] for s in brief["sections"] if s not in sections]
if skipped:
    print(f"Skipping unresearched sections: {skipped}")

,section,chars,sentences,citation_markers,sources,sources_with_claimMatches
0,Educational Outcomes,6915,53,58,161,51
1,School Attendance And Enrollment,5804,39,35,149,34
2,Attentiveness And Learning,4781,29,37,110,31
3,Cognitive Ability And Academic Performance,6542,45,79,109,58


Skipping unresearched sections: ['Nutritional And Health Outcomes', 'Cost-Effectiveness And Efficiency']


## 3. A worked example: passage → cited source text

One citing sentence from the brief, the source record its `[n]` marker resolves to, and —
where the system stored them — the exact spans inside the source excerpt that support the
claim, highlighted in context.

In [4]:
from IPython.display import HTML, display
import html as html_mod


def render_pair(pair: CitationPair, max_excerpt_chars: int = 2500) -> None:
    src = pair.source
    parts = [
        f"<p><b>Section:</b> {html_mod.escape(pair.section_title)}</p>",
        f"<p><b>Brief passage:</b> <i>{html_mod.escape(pair.sentence)}</i></p>",
        f"<p><b>Citation:</b> [{pair.citation_index}]</p>",
    ]
    if src is None:
        parts.append(
            "<p style='color:#b00'><b>DANGLING — no source with this index "
            "exists in the section.</b></p>"
        )
    else:
        breadcrumb, body = parse_section_breadcrumb(src.get("text") or "")
        parts.append(
            f"<p><b>Source:</b> {html_mod.escape(src.get('title') or '')} "
            f"(p. {src.get('page')}, chunk <code>{src.get('chunkId')}</code>)</p>"
        )
        if breadcrumb:
            parts.append(f"<p><b>Location in document:</b> {html_mod.escape(breadcrumb)}</p>")
        spans = sorted(
            (pair.claim_match or {}).get("matches") or [], key=lambda m: m.get("start", 0)
        )
        if spans:
            rendered, cursor = [], 0
            for span in spans:
                start, end = span.get("start", 0), span.get("end", 0)
                rendered.append(html_mod.escape(body[cursor:start]))
                rendered.append(f"<mark>{html_mod.escape(body[start:end])}</mark>")
                cursor = end
            rendered.append(html_mod.escape(body[cursor:]))
            excerpt = "".join(rendered)
            parts.append("<p><b>Source excerpt (supporting spans highlighted):</b></p>")
        else:
            excerpt = html_mod.escape(body)
            parts.append("<p><b>Source excerpt (no stored supporting spans):</b></p>")
        if len(excerpt) > max_excerpt_chars:
            excerpt = excerpt[:max_excerpt_chars] + " …"
        parts.append(
            f"<blockquote style='white-space:pre-wrap'>{excerpt}</blockquote>"
        )
    display(HTML("".join(parts)))


all_pairs = [pair for section in sections for pair in extract_citation_pairs(section)]
print(f"{len(all_pairs)} citation pairs (citing sentence x cited source) in this brief")

example = next((p for p in all_pairs if p.has_stored_support), all_pairs[0])
render_pair(example)

443 citation pairs (citing sentence x cited source) in this brief


## 4. Fidelity signals stored in the system

Three checks, all from data the system already persists:

- **Citation coverage** — how many sentences of each section carry at least one `[n]` marker.
  (Headings and transition sentences legitimately go uncited; factual claims should not.)
- **Dangling citations** — markers whose index has no matching source record. These are
  unverifiable by construction and render as broken citations in the UI.
- **Stored claim support** — citation pairs whose source carries a `claimMatches` entry for
  that exact sentence, i.e. the system has already located supporting text in the source.
  Note this is a *lazy* signal: the UI computes it in the background after research, so
  absence means "not yet checked", not "unsupported".

In [5]:
rows = []
for section in sections:
    markdown = section.get("content") or ""
    sentences = split_sentences(markdown)
    prose = [s for s in sentences if not s.startswith("#")]
    cited = [s for s in prose if extract_cited_numbers(s)]
    pairs = extract_citation_pairs(section)
    source_indices = {
        src["index"] for src in section.get("sources") or [] if src.get("index") is not None
    }
    cited_indices = set(extract_cited_numbers(markdown))
    rows.append(
        {
            "section": section["title"],
            "prose_sentences": len(prose),
            "cited_sentences": len(cited),
            "coverage_%": round(100 * len(cited) / len(prose), 1) if prose else 0.0,
            "citation_pairs": len(pairs),
            "dangling_pairs": sum(1 for p in pairs if p.dangling),
            "pairs_with_stored_support": sum(1 for p in pairs if p.has_stored_support),
            "sources_cited": len(cited_indices & source_indices),
            "sources_retrieved": len(source_indices),
        }
    )
fidelity = pd.DataFrame(rows)
display(fidelity)

dangling = [p for p in all_pairs if p.dangling]
if dangling:
    print(f"\n{len(dangling)} DANGLING citation pair(s) — cited index has no source record:")
    for p in dangling:
        print(f"  - [{p.citation_index}] in section {p.section_title!r}:")
        print(f"      {p.sentence[:160]}")
else:
    print("\nNo dangling citations — every marker resolves to a stored source.")

,section,prose_sentences,cited_sentences,coverage_%,citation_pairs,dangling_pairs,pairs_with_stored_support,sources_cited,sources_retrieved
0,Educational Outcomes,46,41,89.1,91,0,75,58,161
1,School Attendance And Enrollment,35,30,85.7,63,0,56,35,149
2,Attentiveness And Learning,28,23,82.1,43,2,35,36,110
3,Cognitive Ability And Academic Performance,42,42,100.0,246,1,130,78,109



3 DANGLING citation pair(s) — cited index has no source record:
  - [113] in section 'Attentiveness And Learning':
      Teachers and school staff frequently observe that students who receive school meals are more active, energetic, and attentive [40, 113].
  - [113] in section 'Attentiveness And Learning':
      These programs contribute to increased energy and participation in class [20, 113].
  - [145] in section 'Cognitive Ability And Academic Performance':
      This is particularly true for children who were least engaged at baseline [1, 145].


## 5. Passage ↔ source-text table

Every citation pair with the stored evidence: the brief sentence, the cited document/page,
and — where present — the supporting text the system found inside the source excerpt.
Exported as CSV next to the notebook for review outside Jupyter.

In [6]:
pair_rows = []
for p in all_pairs:
    src = p.source or {}
    pair_rows.append(
        {
            "section": p.section_title,
            "brief_sentence": p.sentence,
            "citation": p.citation_index,
            "dangling": p.dangling,
            "doc_title": src.get("title"),
            "page": src.get("page"),
            "chunk_id": src.get("chunkId"),
            "stored_support": p.has_stored_support,
            "supporting_text": " … ".join(p.matched_texts) or None,
            "pdf_url": src.get("pdfUrl"),
        }
    )
pairs_df = pd.DataFrame(pair_rows)

out_dir = REPO_ROOT / "notebooks" / "output"
out_dir.mkdir(exist_ok=True)
csv_path = out_dir / f"citation_pairs_{brief_id}.csv"
pairs_df.to_csv(csv_path, index=False)
print(f"Wrote {len(pairs_df)} pairs to {csv_path}")

display(
    pairs_df[pairs_df["stored_support"]][
        ["section", "brief_sentence", "citation", "doc_title", "page", "supporting_text"]
    ].head(10)
)

Wrote 443 pairs to /Users/jan/Desktop/WFP/evidencelab_environment_20260614_184630/repo/notebooks/output/citation_pairs_190ce56e-cda4-4b71-8d76-8e8e856df39b.csv


,section,brief_sentence,citation,doc_title,page,supporting_text
0,Educational Outcomes,While direct impacts on learning and cognitive abilities can be complex without additi...,26,Strategic evaluation of the contribution of school feeding activities to the achieveme...,52.0,Contributing to learning outcomes: School feeding can contribute to learning outcomes ...
1,Educational Outcomes,While direct impacts on learning and cognitive abilities can be complex without additi...,21,Evaluation of South Sudan WFP Interim Country Strategic Plan 2018-2022,57.0,Number of primary schools assisted (on- sit … Number of students (primary schools) ...
2,Educational Outcomes,While direct impacts on learning and cognitive abilities can be complex without additi...,22,Gambia: School-based Programmes Impact Evaluation,25.0,"child attendance, which is expected to inc … outcomes on school progression, co..."
4,Educational Outcomes,While direct impacts on learning and cognitive abilities can be complex without additi...,37,Strategic evaluation of the contribution of school feeding activities to the achieveme...,52.0,positive effects on enrolment and retention (including for girls) are the most commonl...
5,Educational Outcomes,While direct impacts on learning and cognitive abilities can be complex without additi...,41,Evaluation of National School Feeding Programme in Eswatini (2010-2018),47.0,"benefits that school feeding has on health, ability of children to learn and concentr..."
6,Educational Outcomes,While direct impacts on learning and cognitive abilities can be complex without additi...,43,Operation Evaluations Series - Regional Synthesis 2013-2017 - East and Central Africa ...,21.0,Attendance rates met or exceeded targets
8,Educational Outcomes,While direct impacts on learning and cognitive abilities can be complex without additi...,47,Evaluation of Tsogolo la Thanzi - Healthy Future Home-Grown School Feeding Project in ...,72.0,improving their attendance and concentration … Attendance of learners has improved … s...
11,Educational Outcomes,"School feeding programs can improve learning outcomes, particularly when combined with...",26,Strategic evaluation of the contribution of school feeding activities to the achieveme...,52.0,School feeding can contribute to learning outcomes only when combined with complementa...
12,Educational Outcomes,"In Kenya, school meal programs were linked to improved numeracy [28].",28,Strategic evaluation of the contribution of school feeding activities to the achieveme...,53.0,WFP school feeding programme in Kenya showed that the school meals programme was signi...
13,Educational Outcomes,"Similarly, in Ethiopia, school feeding, along with interventions for girls, enhanced l...",28,Strategic evaluation of the contribution of school feeding activities to the achieveme...,53.0,"ions of Ethiopia demonstrated significant output, outcome and impact level results and..."


## 6. Ground truth: do the stored excerpts match the source documents?

The excerpt in `sources[].text` is what the brief's citations (hover cards, Word-export
reference excerpts) show the reader — so it must equal the actual chunk text extracted from
the source PDF. Compare each cited excerpt against `chunks_<source>.sys_text`, applying the
same `clean_text()` encoding repair the server applies when building sources
(`ui/backend/services/assistant_graph.py`). `sys_text` itself starts with the same
`-- heading --` breadcrumb line as the stored excerpt, so the two are compared in full.

In [7]:
import difflib

cited_sources = {}
for p in all_pairs:
    if p.source and p.source.get("chunkId"):
        cited_sources[(p.section_title, p.source["chunkId"])] = p.source

chunk_ids = sorted({chunk_id for _, chunk_id in cited_sources})
with conn.cursor() as cur:
    cur.execute(
        f"SELECT chunk_id, doc_id, sys_page_num, sys_text FROM {chunks_table} "
        "WHERE chunk_id = ANY(%s)",
        (chunk_ids,),
    )
    db_chunks = {str(r[0]): {"doc_id": str(r[1]), "page": r[2], "sys_text": r[3]} for r in cur}

verify_rows = []
for (section_title, chunk_id), src in sorted(cited_sources.items()):
    db = db_chunks.get(chunk_id)
    if db is None:
        verify_rows.append(
            {"section": section_title, "chunk_id": chunk_id, "status": "MISSING_IN_DB",
             "similarity": 0.0}
        )
        continue
    # Both the stored excerpt and sys_text carry the leading breadcrumb line,
    # so compare them in full (whitespace-insensitive).
    stored = normalize_ws(src.get("text") or "")
    truth = normalize_ws(clean_text(db["sys_text"] or ""))
    if stored == truth:
        status, ratio = "exact", 1.0
    else:
        ratio = difflib.SequenceMatcher(None, stored, truth).ratio()
        status = "drift" if ratio < 0.995 else "whitespace/encoding"
    verify_rows.append(
        {"section": section_title, "chunk_id": chunk_id, "status": status,
         "similarity": round(ratio, 4)}
    )

verify_df = pd.DataFrame(verify_rows)
print(f"{len(verify_df)} distinct cited (section, chunk) excerpts checked against {chunks_table}:")
display(verify_df["status"].value_counts().rename_axis("status").to_frame("chunks"))
problems = verify_df[verify_df["status"].isin(["MISSING_IN_DB", "drift"])]
if len(problems):
    display(problems)
else:
    print("Every cited excerpt matches the ground-truth chunk text in the database.")

207 distinct cited (section, chunk) excerpts checked against chunks_wfp:


,chunks
status,
exact,177
whitespace/encoding,30


Every cited excerpt matches the ground-truth chunk text in the database.


## 7. Optional: re-run the system's claim-support check for unchecked pairs

`claimMatches` only exist where the UI's background highlight pass has run. For citation pairs
without stored spans, the notebook can ask the system itself whether the source excerpt
supports the sentence — the same `POST /highlight` (semantic, LLM-based) call the UI makes,
with the same payload shape (`briefHighlights.ts` → `findSemanticMatches`). The
`semantic_model_config` is required (the server has no default LLM and silently returns zero
matches without it) and is taken from `config.json` → `ui_model_combos[<combo>].semantic_highlighting_model`,
exactly where the UI gets it (`App.tsx`).

Each call is one LLM round-trip (~10s), so this runs on a small sample by default. Raise
`LIVE_CHECK_SAMPLE` (or set it to `None` for *all* unchecked pairs) for a fuller audit.

In [8]:
import requests

RUN_LIVE_HIGHLIGHT = True
LIVE_CHECK_SAMPLE = 3  # pairs to check; None = all unchecked pairs
SEMANTIC_THRESHOLD = 0.4  # UI default
MIN_MATCH_CHARS = 30  # briefHighlights.ts drops shorter fragments as misleading

# Same source the UI uses for its highlight model. Pick another combo from
# config.json (or set EVIDENCELAB_MODEL_COMBO) if this one has no working
# credentials locally.
combos = json.loads((REPO_ROOT / "config.json").read_text())["ui_model_combos"]
MODEL_COMBO = os.environ.get("EVIDENCELAB_MODEL_COMBO", next(iter(combos)))
SEMANTIC_MODEL_CONFIG = combos[MODEL_COMBO]["semantic_highlighting_model"]
print(f"Highlight model: {SEMANTIC_MODEL_CONFIG['model']} (combo {MODEL_COMBO!r})")


def live_claim_support(sentence: str, excerpt_body: str) -> list:
    """Ask POST /highlight for spans of `excerpt_body` supporting `sentence`."""
    response = requests.post(
        f"{API_BASE}/highlight",
        headers={"X-API-Key": API_KEY},
        json={
            "query": normalize_claim_text(sentence),
            "text": excerpt_body,
            "highlight_type": "semantic",
            "semantic_threshold": SEMANTIC_THRESHOLD,
            "semantic_model_config": SEMANTIC_MODEL_CONFIG,
        },
        timeout=300,
    )
    response.raise_for_status()
    matches = response.json().get("matches") or []
    return [m for m in matches if m.get("end", 0) - m.get("start", 0) >= MIN_MATCH_CHARS]


live_results = []
if RUN_LIVE_HIGHLIGHT:
    unchecked = [p for p in all_pairs if not p.dangling and not p.has_stored_support]
    sample = unchecked if LIVE_CHECK_SAMPLE is None else unchecked[:LIVE_CHECK_SAMPLE]
    print(f"{len(unchecked)} unchecked pairs; live-checking {len(sample)} of them…")
    for p in sample:
        _, body = parse_section_breadcrumb(p.source.get("text") or "")
        matches = live_claim_support(p.sentence, body)
        live_results.append({"pair": p, "matches": matches})
        verdict = "SUPPORTED" if matches else "no supporting span found"
        print(f"\n[{p.citation_index}] {p.sentence[:120]}…" if len(p.sentence) > 120
              else f"\n[{p.citation_index}] {p.sentence}")
        print(f"  → {verdict}")
        for m in matches:
            text = body[m["start"]:m["end"]]
            print(f"    supporting text: {text[:200]!r}")
else:
    print("Live check skipped (RUN_LIVE_HIGHLIGHT = False).")

Highlight model: gpt-4.1-mini (combo 'Azure Foundry')
144 unchecked pairs; live-checking 3 of them…



[29] While direct impacts on learning and cognitive abilities can be complex without additional interventions, the benefits f…
  → no supporting span found



[44] While direct impacts on learning and cognitive abilities can be complex without additional interventions, the benefits f…
  → no supporting span found

[51] While direct impacts on learning and cognitive abilities can be complex without additional interventions, the benefits f…
  → no supporting span found


## 8. RAGAS faithfulness — quantified, claim-level groundedness

The checks so far *locate* evidence; they don't *judge* whether each fact is actually
entailed by it. [RAGAS](https://docs.ragas.io)'s **Faithfulness** metric closes that gap: an
LLM judge decomposes each section into atomic factual claims, then runs an NLI check of every
claim against the section's cited source texts. Score = supported claims ÷ total claims, and
every claim comes back with a verdict **and a written reason** — so a human only needs to
review the claims judged unsupported, not all of them.

**Judge LLM and credentials** — nothing new is configured:

- Model: `gpt-4.1-mini` on **Azure Foundry**, the same deployment the app itself uses (the
  `"Azure Foundry"` combo in `config.json`).
- Credentials: `AZURE_FOUNDRY_KEY` / `AZURE_FOUNDRY_ENDPOINT` (+ optional
  `AZURE_FOUNDRY_API_VERSION`) from the repo `.env` — identical to
  `utils/llm_factory.py`'s `azure_foundry` provider, including the
  `…/openai/deployments/<model>` base-URL scheme. (RAGAS 0.4's modern metrics API takes an
  `openai.AsyncOpenAI` client rather than the factory's LangChain object; the legacy
  LangChain wrapper path is deprecated in RAGAS and rejected by these metrics.)
- No embedding model is needed for Faithfulness.

**Semantics to keep in mind:** RAGAS judges each claim against the *union* of the section's
cited chunks ("bag of context"). That answers *"is the brief grounded in what it cites?"* —
hallucination detection. It does **not** check that the specific `[n]` next to a sentence
points at the right chunk; that per-citation precision check is the DeepEval follow-up.

Cost: per section, 1 claim-decomposition call plus 1 NLI verdict call per 10 claims. Each NLI
call carries all cited chunks (tens of thousands of tokens for heavily-cited sections), well
within gpt-4.1-mini's context window.

In [9]:
RUN_RAGAS = True

from openai import AsyncOpenAI
from ragas.llms.base import llm_factory as ragas_llm_factory
from ragas.metrics.collections import Faithfulness

RAGAS_MODEL = "gpt-4.1-mini"  # Azure Foundry deployment name, as in config.json

if RUN_RAGAS:
    endpoint = DOTENV["AZURE_FOUNDRY_ENDPOINT"].rstrip("/")
    # Same URL scheme as utils/llm_factory.py::_create_azure_foundry_llm.
    if "/openai/deployments" not in endpoint:
        endpoint = f"{endpoint}/openai/deployments/{RAGAS_MODEL}"
    judge_client = AsyncOpenAI(
        api_key=DOTENV["AZURE_FOUNDRY_KEY"],
        base_url=endpoint,
        default_query={
            "api-version": DOTENV.get("AZURE_FOUNDRY_API_VERSION", "2024-02-15-preview")
        },
    )
    # Generous output budget: the NLI verdict responses carry a written reason
    # per claim, and a too-small cap raises IncompleteOutputException
    # (config.json allows up to 32768 output tokens for this model).
    judge_llm = ragas_llm_factory(RAGAS_MODEL, client=judge_client, max_tokens=16000)
    faithfulness = Faithfulness(llm=judge_llm)
    print(f"RAGAS judge ready: {RAGAS_MODEL} @ {endpoint.split('/openai/')[0]}")
else:
    print("RAGAS skipped (RUN_RAGAS = False).")

RAGAS judge ready: gpt-4.1-mini @ https://evidencelab-resource.openai.azure.com


In [10]:
ragas_scores = pd.DataFrame()
ragas_claims = pd.DataFrame()

# Statements judged per NLI call. One call for a whole section (RAGAS's default)
# can exceed the judge's output budget once reasons are included; small batches
# keep each response comfortably inside it.
VERDICT_BATCH = 10

if RUN_RAGAS:
    score_rows, claim_rows = [], []
    for section in sections:
        sample = build_faithfulness_input(brief["query"], section)
        statements = await faithfulness._create_statements(
            sample["user_input"], sample["response"]
        )
        context_str = "\n".join(sample["retrieved_contexts"])
        judged = []
        for at in range(0, len(statements), VERDICT_BATCH):
            batch = statements[at : at + VERDICT_BATCH]
            verdicts = await faithfulness._create_verdicts(batch, context_str)
            judged.extend(verdicts.statements)
        supported_n = sum(1 for s in judged if s.verdict)
        score = supported_n / len(judged) if judged else float("nan")
        score_rows.append(
            {
                "section": section["title"],
                "claims": len(judged),
                "supported": supported_n,
                "faithfulness": round(score, 3),
                "contexts": len(sample["retrieved_contexts"]),
            }
        )
        for s in judged:
            claim_rows.append(
                {
                    "section": section["title"],
                    "claim": s.statement,
                    "supported": bool(s.verdict),
                    "judge_reason": s.reason,
                }
            )
        print(
            f"{section['title']!r}: faithfulness={score:.3f} "
            f"({supported_n}/{len(judged)} claims supported)"
        )

    ragas_scores = pd.DataFrame(score_rows)
    ragas_claims = pd.DataFrame(claim_rows)
    ragas_claims.to_csv(out_dir / f"ragas_claims_{brief_id}.csv", index=False)
    total_claims = int(ragas_scores["claims"].sum())
    total_supported = int(ragas_scores["supported"].sum())
    print(
        f"\nBrief-level faithfulness: {total_supported}/{total_claims} claims supported "
        f"({total_supported / total_claims:.1%})"
    )
    display(ragas_scores)

'Educational Outcomes': faithfulness=0.951 (97/102 claims supported)


'School Attendance And Enrollment': faithfulness=0.948 (92/97 claims supported)


'Attentiveness And Learning': faithfulness=1.000 (39/39 claims supported)


'Cognitive Ability And Academic Performance': faithfulness=0.971 (67/69 claims supported)

Brief-level faithfulness: 295/307 claims supported (96.1%)


,section,claims,supported,faithfulness,contexts
0,Educational Outcomes,102,97,0.951,58
1,School Attendance And Enrollment,97,92,0.948,35
2,Attentiveness And Learning,39,39,1.000,36
3,Cognitive Ability And Academic Performance,69,67,0.971,78


### Claims the judge could NOT support — the human review queue

These are the only claims a reviewer still has to check by hand (plus a small random sample
of supported ones, if you want to validate the judge itself). Each comes with the judge's
reason, and the passage table from section 5 gives the citation and PDF link to verify
against.

In [11]:
if RUN_RAGAS:
    unsupported = ragas_claims[~ragas_claims["supported"]]
    print(
        f"{len(unsupported)} of {len(ragas_claims)} claims judged unsupported "
        f"by the cited sources:"
    )
    with pd.option_context("display.max_colwidth", 160):
        display(unsupported[["section", "claim", "judge_reason"]])
else:
    print("RAGAS skipped (RUN_RAGAS = False).")

12 of 307 claims judged unsupported by the cited sources:


,section,claim,judge_reason
3,Educational Outcomes,School feeding programs influence cognitive development.,"The context indicates that while school feeding may improve attentiveness and concentration, evaluations found no significant direct impacts on cognitive ab..."
4,Educational Outcomes,School feeding programs influence academic performance.,The context shows that school feeding alone does not conclusively improve academic performance or standardized test scores; improvements are limited by stru...
34,Educational Outcomes,"In Rwanda, the average student attendance rate increased from 93% to 94.3% with nutritious school meals.","The context mentions an increase in average student attendance rate from 93% to 94.3% in FY 2022 to FY 2023, but this data is not specifically attributed to..."
37,Educational Outcomes,School meals in Guinea-Bissau especially promote attendance and reduce dropout rates for girls.,"The context provides disaggregated data showing retention rates for girls and boys are similar and high, and mentions positive impacts on attendance and dro..."
56,Educational Outcomes,Some studies have found negative impacts on cognitive abilities due to school feeding.,"The context does not provide evidence of negative impacts on cognitive abilities; rather, it reports null or no significant effects, but not negative impacts."
110,School Attendance And Enrollment,"In Mali, children from households receiving school meals were 10 percentage points more likely to be enrolled in school.",The context does not mention Mali or provide specific data about enrollment increases of 10 percentage points in Mali; the 10 percentage points figure is me...
111,School Attendance And Enrollment,"In Mali, children from households receiving school meals completed nearly half an additional year of education.",The context does not mention Mali or provide data about children completing nearly half an additional year of education in Mali; this specific statistic is ...
127,School Attendance And Enrollment,"In Laos, an impact assessment revealed an average enrollment increase of 5.3 percent in program schools.","The context does not provide any information about Laos or enrollment changes in Laos; therefore, this statement cannot be directly inferred."
128,School Attendance And Enrollment,"In Laos, control schools experienced a 2.0 percent decrease in enrollment.","The context does not mention Laos or enrollment changes in control schools in Laos; thus, this statement cannot be directly inferred."
169,School Attendance And Enrollment,Complementary interventions play a significant role in the effectiveness of school feeding programs.,"The context mentions that except for schools benefiting from literacy components, no significant differences were found based on complementary activities, s..."


## 9. Summary

In [12]:
total = len(all_pairs)
supported = sum(1 for p in all_pairs if p.has_stored_support)
dangling_n = sum(1 for p in all_pairs if p.dangling)
live_supported = sum(1 for r in live_results if r["matches"])
excerpts_ok = int(verify_df["status"].isin(["exact", "whitespace/encoding"]).sum())

if RUN_RAGAS and len(ragas_scores):
    _claims = int(ragas_scores["claims"].sum())
    _supported = int(ragas_scores["supported"].sum())
    ragas_line = (
        f"RAGAS faithfulness ({RAGAS_MODEL} judge): {_supported}/{_claims} atomic claims "
        f"supported ({_supported / _claims:.1%}); "
        f"{_claims - _supported} claims for human review."
    )
else:
    ragas_line = "RAGAS faithfulness: not run this session (RUN_RAGAS = False)."

print(f"""Citation fidelity — {brief_title!r}

Citation pairs (citing sentence x cited source):  {total}
  with stored supporting spans (claimMatches):    {supported}
  dangling (marker resolves to no source):        {dangling_n}
  live-checked this run:                          {len(live_results)} ({live_supported} supported)
  unchecked (no stored spans, not sampled):       {total - supported - dangling_n - len(live_results)}

Ground truth: {excerpts_ok}/{len(verify_df)} cited excerpts match chunks table text.

{ragas_line}

Interpretation:
- Every non-dangling citation links to a stored source record that carries the FULL text of
  the source-document chunk, so every cited fact is verifiable against real source text.
- 'Stored supporting spans' means the system has already pinpointed the sentence-level
  evidence; absence is 'not yet checked' (the UI backfills these lazily), not 'unsupported'.
- Dangling markers are genuine fidelity defects: the reader is shown a citation that cannot
  be resolved to any source.
- RAGAS judges claims against the UNION of a section's cited chunks: it quantifies
  groundedness (hallucination risk), not whether each [n] points at the right chunk.
  Its unsupported-claims list (section 8) is the human review queue.
- To audit every unchecked pair, set LIVE_CHECK_SAMPLE = None in section 7 and re-run.""")

conn.close()

Citation fidelity — 'School Feeding Programmes: Evidence On Learning, Nutrition, And Local Procurement Outcomes'

Citation pairs (citing sentence x cited source):  443
  with stored supporting spans (claimMatches):    296
  dangling (marker resolves to no source):        3
  live-checked this run:                          3 (0 supported)
  unchecked (no stored spans, not sampled):       141

Ground truth: 207/207 cited excerpts match chunks table text.

RAGAS faithfulness (gpt-4.1-mini judge): 295/307 atomic claims supported (96.1%); 12 claims for human review.

Interpretation:
- Every non-dangling citation links to a stored source record that carries the FULL text of
  the source-document chunk, so every cited fact is verifiable against real source text.
- 'Stored supporting spans' means the system has already pinpointed the sentence-level
  evidence; absence is 'not yet checked' (the UI backfills these lazily), not 'unsupported'.
- Dangling markers are genuine fidelity defects: the r